# Benchmarking tree ensemble formulations

*Note: the formulations compared in this notebook are **experimental**: their
names, keyword arguments and variables may change in future releases.*

*All the models in this notebook have fewer than 2000 variables and 2000
constraints, so they can be solved with a size-limited license.*

Which formulation of a tree ensemble solves fastest depends on the ensemble
and on the optimization problem around it. This notebook defines a small
function that builds and solves a model with each formulation and reports the
model size and the build and solve times. The example is deliberately small;
replace the regression and the model with your own to compare the formulations
on your problem.

The formulations are described in the
[Mixed Integer Formulations](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/user/mip-models.html)
page of the user guide, and the notebook *Tree Ensemble Formulations* shows
how they differ on a small example.

In [ ]:
import time

import gurobipy as gp
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor

from gurobi_ml import add_predictor_constr

FORMULATIONS = ["leaf", "misic", "misic_lazy", "ocean", "biggs_perakis"]

## The model to compare on

We use the same example as in *Tree Ensemble Formulations*: a gradient boosting
regressor fitted on the diabetes dataset of scikit-learn, and a model that
maximizes its prediction over the range of the data.

`build_model` takes the name of a formulation and returns a model ready to be
solved. This is the function to replace with your own model: the benchmark
below only needs a model built with the given formulation.

In [ ]:
X, y = load_diabetes(return_X_y=True)
feature_lb = X.min(axis=0)
feature_ub = X.max(axis=0)

regression = GradientBoostingRegressor(n_estimators=20, max_depth=3, random_state=0)
regression.fit(X, y)


def build_model(formulation):
    m = gp.Model()
    m.Params.OutputFlag = 0
    x = m.addMVar((1, X.shape[1]), lb=feature_lb, ub=feature_ub, name="x")
    y_pred = m.addMVar((1, 1), lb=-gp.GRB.INFINITY, name="y")
    add_predictor_constr(m, regression, x, y_pred, formulation=formulation)
    m.setObjective(y_pred.sum(), gp.GRB.MAXIMIZE)
    return m

## The benchmark function

`benchmark` builds and solves the model once per formulation and per value of
Gurobi's `Seed` parameter. Solving with several seeds shows how much the solve
time varies from run to run, which for MIPs can be large. Further Gurobi
parameters, for example `TimeLimit` or `Threads`, are passed as keyword
arguments.

Each row of the result is one solve. The times are measured in seconds, both
as wall-clock time and as CPU time of the Python process, which includes the
threads of Gurobi.

In [ ]:
def benchmark(build_model, formulations, seeds, **params):
    rows = []
    for formulation in formulations:
        for seed in seeds:
            wall, cpu = time.perf_counter(), time.process_time()
            m = build_model(formulation)
            m.update()
            build_wall = time.perf_counter() - wall
            build_cpu = time.process_time() - cpu

            for name, value in params.items():
                m.setParam(name, value)
            m.Params.Seed = seed
            wall, cpu = time.perf_counter(), time.process_time()
            m.optimize()
            rows.append(
                {
                    "formulation": formulation,
                    "seed": seed,
                    "variables": m.NumVars,
                    "binaries": m.NumBinVars,
                    "linear constraints": m.NumConstrs,
                    "indicator constraints": m.NumGenConstrs,
                    "build wall [s]": build_wall,
                    "build CPU [s]": build_cpu,
                    "solve wall [s]": time.perf_counter() - wall,
                    "solve CPU [s]": time.process_time() - cpu,
                    "nodes": m.NodeCount,
                    "status": m.Status,
                    "objective": m.ObjVal if m.SolCount > 0 else float("nan"),
                }
            )
            m.dispose()
    return pd.DataFrame(rows)

## Run it

We solve each formulation with three seeds.

In [ ]:
results = benchmark(build_model, FORMULATIONS, seeds=range(3))
results

The table below summarizes the solves of each formulation: the model size,
the median times and number of nodes, and the range of the objective values.

In [ ]:
summary = results.groupby("formulation", sort=False).agg(
    variables=("variables", "first"),
    binaries=("binaries", "first"),
    constraints=("linear constraints", "first"),
    indicators=("indicator constraints", "first"),
    build_wall=("build wall [s]", "median"),
    solve_wall=("solve wall [s]", "median"),
    solve_cpu=("solve CPU [s]", "median"),
    nodes=("nodes", "median"),
    objective_min=("objective", "min"),
    objective_max=("objective", "max"),
)
summary

## Reading the results

This model solves in a fraction of a second with every formulation, so the
times above cannot tell the formulations apart. To compare them on your
problem:

- Replace `regression` and `build_model` with your own ensemble and model,
  including the constraints of your decision problem. The ranking of the
  formulations changes with the problem, and maximizing a prediction over a
  box is not representative of most applications.
- Solve with several seeds and compare medians: the solve time of a MIP can
  vary a lot between seeds.
- Pass the parameters you use in practice, for example
  `benchmark(build_model, FORMULATIONS, seeds=range(5), TimeLimit=600, Threads=4)`.
- Compare the objective values, not only the times. With the default
  `epsilon=0`, `"leaf"`, `"ocean"` and `"biggs_perakis"` may report a value that
  no input attains, as shown in *Tree Ensemble Formulations*.
- `"ocean"` and `"biggs_perakis"` require finite bounds on the input variables
  used in splits, and raise an error otherwise.

Copyright © 2023-2026 Gurobi Optimization, LLC